In [ ]:
##Classifying Bilateral Anchor currencies
ssa = [
    "AGO","BEN","BWA","BFA","BDI","CPV","CMR","CAF","TCD","COM",
    "COG","COD","CIV","DJI","GNQ","ERI","SWZ","ETH","GAB","GMB",
    "GHA","GIN","GNB","KEN","LSO","LBR","MDG","MWI","MLI","MRT",
    "MUS","MOZ","NAM","NER","NGA","RWA","STP","SEN","SYC","SLE",
    "SOM","ZAF","TZA","TGO","UGA","ZMB","ZWE"]

anchor_countries = ["FRA", "GBR", "USA", "ESP", "PRT", "BEL", "ZAR", "DEM"]

# Map currency labels to country columns in the source data
source_code = {"ZAR": "ZAF", "DEM": "DEU"}

countries = list(dict.fromkeys(
    ssa + ["FRA", "GBR", "USA", "ESP", "PRT", "BEL", "DEU"]))

regime = regime_df.loc["1962M1":"1983M12", countries]
anchor = anchor_df.loc["1962M1":"1983M12", countries]

regime.index = pd.to_datetime(regime.index, format="%YM%m")
anchor.index = pd.to_datetime(anchor.index, format="%YM%m")

regime_yearly = regime.groupby(regime.index.year).agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)

anchor_yearly = anchor.groupby(anchor.index.year).agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)

regime_yearly.index.name = "year"
anchor_yearly.index.name = "year"

pairs = list(combinations(anchor_countries, 2))

# Currency label -> country column in source data
currency_source = {"FRF": "FRA","GBP": "GBR", "USD": "USA","ESP": "ESP","PTE": "PRT","BEF": "BEL",
    "ZAR": "ZAF","DEM": "DEU"}

def anchors_connected(year, a1, a2):
    if pd.isna(a1) or pd.isna(a2):
        return False

    a1, a2 = str(a1).strip(), str(a2).strip()

    if a1 == a2:
        return True

    # Construct the year's currency-peg network
    graph = {}

    for currency, country_col in currency_source.items():
        anchor_value = anchor_yearly.loc[year, country_col]

        if pd.isna(anchor_value):
            continue

        anchor_value = str(anchor_value).strip()

        graph.setdefault(currency, set()).add(anchor_value)
        graph.setdefault(anchor_value, set()).add(currency)

    # Search for any direct or indirect path
    stack = [a1]
    visited = set()

    while stack:
        node = stack.pop()

        if node == a2:
            return True

        if node in visited:
            continue

        visited.add(node)
        stack.extend(graph.get(node, set()) - visited)

    return False


results = []

for year in sorted(set(regime_yearly.index) & set(anchor_yearly.index)):
    for c1, c2 in pairs:

        col1 = source_code.get(c1, c1)
        col2 = source_code.get(c2, c2)

        r1 = regime_yearly.loc[year, col1]
        r2 = regime_yearly.loc[year, col2]

        a1 = anchor_yearly.loc[year, col1]
        a2 = anchor_yearly.loc[year, col2]

        fixed = (r1 in [1, 2, 6] and r2 in [1, 2, 6] and pd.notna(a1) and pd.notna(a2) and 
                 anchors_connected(year, a1, a2))

        results.append({"year": year,"country1": c1,"country2": c2,"anchor1": a1,"anchor2": a2,"fixed": int(fixed)
        })

pair_fixed = pd.DataFrame(results)

path = r"anchor_country_pair_fixed_dummy_1962_1983.csv"

pair_fixed.to_csv(path, index=False)

In [ ]:
##Classifying sub-Saharan African pairs

regime = regime_df.loc["1963M1":"1982M12", ssa].copy()
anchor = anchor_df.loc["1963M1":"1982M12", ssa].copy()

regime.index = pd.to_datetime(regime.index, format="%YM%m")
anchor.index = pd.to_datetime(anchor.index, format="%YM%m")

regime_yearly = regime.groupby(regime.index.year).agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)

anchor_yearly = anchor.groupby(anchor.index.year).agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)

xof_members = ["BEN","BFA","CIV","NER","SEN","TGO"]
xaf_members = ["CMR","CAF","TCD","COG","GAB"]
eas_members = ["KEN","TZA","UGA"]

def get_cfa_region(iso, year):
    if iso in xof_members:
        return "XOF"
    elif iso == "MRT" and year <= 1972:
        return "XOF"
    elif iso == "MLI" and (year <= 1962 or year >= 1983):
        return "XOF"
    elif iso in xaf_members:
        return "XAF"
    return np.nan

results = []

for year in range(1963, 1983):

    for iso_o, iso_d in combinations(ssa, 2):

        r_o = regime_yearly.loc[year, iso_o]
        r_d = regime_yearly.loc[year, iso_d]

        a_o = anchor_yearly.loc[year, iso_o]
        a_d = anchor_yearly.loc[year, iso_d]

        # Common currency 
        #CFA West and Central 
        
        cfa_o = get_cfa_region(iso_o, year)
        cfa_d = get_cfa_region(iso_d, year)

        common_xof = int(cfa_o == "XOF" and cfa_d == "XOF")
        common_xaf = int(cfa_o == "XAF" and cfa_d == "XAF")
        
        #East Africa Shilling
        common_eas = int(iso_o in eas_members and iso_d in eas_members and year <= 1967)

        common_currency = int(common_xof or common_xaf or common_eas)

        # Fixed classification
        if common_currency == 1:
            fixed = 0
            
        elif r_o == 5 or r_d == 5:
            fixed = "freely falling"

        elif a_o == "SDR" or a_d == "SDR":
            fixed = 0

        else:
            valid_regime = (pd.notna(r_o) and pd.notna(r_d) and r_o in [1, 2] and r_d in [1, 2])

            if pd.notna(a_o) and pd.notna(a_d):
                c1 = anchor_to_country.get(a_o, a_o)
                c2 = anchor_to_country.get(a_d, a_d)

                anchor_pair_fixed = ( a_o == a_d or
                    anchor_fixed_lookup.get((year, frozenset([c1, c2])), 0) == 1)
            else:
                anchor_pair_fixed = False

            fixed = int(valid_regime and anchor_pair_fixed)
        
        results.append({"year": year, "country1": iso_o,"country2": iso_d,
            "fixed": fixed,"common_currency": common_currency})

ssa_pairs = pd.DataFrame(results)
path = r"ssa_bilateral_fixed_common_currency_1963_1982.csv"
ssa_pairs.to_csv(path, index=False)

In [ ]:
#Constructing the panel from TRADHIST
tradhist["FLOW"] = tradhist["FLOW"].where(
    ~(tradhist["FLOW"].isna() & tradhist["FLOW_0"].eq(0)), 0)
tradhist["year"] = pd.to_numeric(tradhist["year"], errors="coerce")

start_year = 1963
end_year = 1982

# Construct SSA panel
panel = (tradhist[(tradhist["REGIO_o"] == "STHAFR") & (tradhist["REGIO_d"] == "STHAFR") &
        (tradhist["year"].between(start_year, end_year))]
    [["iso_o", "iso_d", "year", "FLOW", "GDP_o", "GDP_d",
      "POP_o", "POP_d", "Distw", "Comlang", "Contig",
      "XCH_RATE_o", "XCH_RATE_d"]].copy())

#1972 indicator
panel["post"] = (panel["year"] > 1972).astype(int)

# Sorted, unordered pair ID
panel["pair_ID"] = panel.apply(lambda r: "_".join(sorted([str(r["iso_o"]), str(r["iso_d"])])),
    axis=1)

countries = set(panel["iso_o"].dropna()) | set(panel["iso_d"].dropna())

In [ ]:
#Oil pair countries
african_oil_producers_1963_1982 = [("AGO", 1955), ("GAB", 1957), ("COG", 1957), ("DZA", 1958), ("NGA", 1958),
    ("TUN", 1966), ("COD", 1975), ("CMR", 1977),("CIV", 1980),]

oil_start = dict(african_oil_producers_1963_1982)

panel["oil_pair"] = (panel.apply(lambda r: (
            r["iso_o"] in oil_start and r["year"] >= oil_start[r["iso_o"]]) or (
            r["iso_d"] in oil_start and r["year"] >= oil_start[r["iso_d"]]),
        axis=1)).astype(int)

print(panel["oil_pair"].value_counts())

panel.to_csv(r"\panel.csv", index=False)

In [ ]:
#assigning tariff agreements information to panel
ta["entryforceyear"] = pd.to_numeric(ta["entryforceyear"], errors="coerce")

ta = ta[["iso_o", "iso_d", "entryforceyear"]].dropna(subset=["entryforceyear"]).copy()

ta = ta[ta["iso_o"] != ta["iso_d"]].copy()

# Create sorted unordered pair_ID
ta["pair_ID"] = ta.apply(lambda r: "_".join(sorted([str(r["iso_o"]), str(r["iso_d"])])),axis=1)

# Keep the earliest entry-into-force year per pair
ta_ordered = (ta.groupby("pair_ID", as_index=False)["entryforceyear"].min())

# Create the same sorted pair_ID in panel
panel["pair_ID"] = panel.apply(lambda r: "_".join(sorted([str(r["iso_o"]), str(r["iso_d"])])),axis=1)

# Join trade agreement data to panel
panel = panel.drop(columns=["entryforceyear", "trade_agreement"], errors="ignore")

panel = panel.merge(ta_ordered,on="pair_ID",how="left",validate="m:1")

# Create dummy: 1 from entry year onward, otherwise 0
panel["trade_agreement"] = (panel["entryforceyear"].notna() &
    (panel["year"] >= panel["entryforceyear"])).astype(int)


In [ ]:
#5-year panel (trade average: main specification)
import wbdata
cpi = wbdata.get_dataframe({"FP.CPI.TOTL": "cpi_2010"},country="USA",date=("1962", "1984")
).reset_index()

cpi["year"] = cpi["date"].astype(int)

cpi = cpi[["year", "cpi_2010"]]
panel_5_3 = panel.merge(cpi, on="year", how="left")

panel_5_3["FLOW"] = (panel_5_3["FLOW"] * 100 / panel_5_3["cpi_2010"])
start_year = 1963

panel_5_3["period_start"] = (start_year + 5 * ((panel_5_3["year"] - start_year) // 5))

panel_5_3["period_end"] = panel_5_3["period_start"] + 4

panel_5_3["period"] = (panel_5_3["period_start"].astype(str)
    + "-"+ panel_5_3["period_end"].astype(str).str[-2:])

panel_5_3["post"] = (panel_5_3["period_start"] > 1972).astype(int)
panel_5_3 = (panel_5_3.groupby(
        ["pair_ID", "iso_o", "iso_d", "period", "period_start"],
        as_index=False).agg(trade_avg=("FLOW", lambda x: 0 if (x <= 0).all()
                   else x[x > 0].mean()),

        # Continuous variables: mean
        Distw=("Distw", "mean"),

        # Dummies: max
        common_currency=("common_currency", max), reg_o=("reg_o",max),
        reg_d=("reg_d", max), anchor_o=("anchor_o", max), anchor_d=("anchor_d", max),
        oil_pair=("oil_pair", max), fixed=("fixed", max),
        trade_agreement=("trade_agreement", max), Contig=("Contig", max),
        Comlang=("Comlang", max)))

panel_5_3["post"] = (panel_5_3["period_start"] > 1972).astype(int)
panel_5_3.to_csv(r"panel_5_3.csv",index=False)

In [ ]:
#5-year panel (robustness check panel with simple average)

panel = pd.read_csv(r"panel.csv")

cpi = wbdata.get_dataframe({"FP.CPI.TOTL": "cpi_2010"},
    country="USA", date=("1962", "1984")).reset_index()

cpi["year"] = cpi["date"].astype(int)
cpi = cpi[["year", "cpi_2010"]]

panel_5_3_simple = panel.merge(cpi, on="year", how="left")

panel_5_3_simple["FLOW"] = ( panel_5_3_simple["FLOW"] * 100/ panel_5_3_simple["cpi_2010"])

start_year = 1963

panel_5_3_simple["period_start"] = ( start_year + 5 * (
        (panel_5_3_simple["year"] - start_year) // 5))

panel_5_3_simple["period_end"] = (
    panel_5_3_simple["period_start"] + 4)

panel_5_3_simple["period"] = (panel_5_3_simple["period_start"].astype(str)+ "-"
    + panel_5_3_simple["period_end"].astype(str).str[-2:])

panel_5_3_simple = (panel_5_3_simple.groupby(
        ["pair_ID", "iso_o", "iso_d", "period", "period_start"],
        as_index=False )
    .agg( # Simple mean: zeros are included
        trade_avg=("FLOW", "mean"), Distw=("Distw", "mean"),
        common_currency=("common_currency", "max"),reg_o=("reg_o", "max"),
        reg_d=("reg_d", "max"), anchor_o=("anchor_o", "max"),
        anchor_d=("anchor_d", "max"), oil_pair=("oil_pair", "max"),
        fixed=("fixed", "max"),trade_agreement=("trade_agreement", "max"),
        Contig=("Contig", "max"),Comlang=("Comlang", "max")))

panel_5_3_simple["post"] = (panel_5_3_simple["period_start"] > 1972).astype(int)

panel_5_3_simple.to_csv(panel_5_3_simple_path, index=False)

print("Saved:", r"panel_5_3_simple.csv")